# Baseline Cardiac Cine MRI Segmentation (2D U-Net)

This notebook implements and demonstrates the **supervised baseline cardiac segmentation pipeline** for the Motion-Guided Self-Supervised Learning project.

### 4-Class Cardiac Segmentation Formulation
- **Class 0**: Background
- **Class 1**: Left Ventricle (LV) Cavity
- **Class 2**: Myocardium
- **Class 3**: Right Ventricle (RV) Cavity

### Compute Constraint Notice
> **IMPORTANT**: This development environment is strictly used for pipeline construction, verification, and lightweight smoke testing. **No model training is performed here**. Full GPU training is executed on the separate training cluster.

In [ ]:
import sys
import os
import json
import yaml
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from torch.utils.data import DataLoader

# Ensure project root is in sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.dataset import ACDCSegDataset, get_train_transforms, get_val_transforms
from src.segmentation_model import build_segmentation_model, SegmentationUNet
from src.encoder import count_parameters
from src.losses import DiceCELoss
from src.metrics import compute_metrics_single, CLASS_NAMES
from src.train import set_seed, get_device

# Load baseline configuration
config_path = os.path.join(PROJECT_ROOT, 'configs', 'baseline_config.yaml')
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

SEED = config.get('random_seed', 42)
set_seed(SEED)

DEVICE = torch.device('cpu')  # Verified CPU mode for smoke test
print(f"Loaded config: {config_path}")
print(f"Active device: {DEVICE}")

## 1. Dataset Loading (Patient-Level Splits)

Loads preprocessed 2D slices (`data/processed/*.npz`) using the patient-level split files (`data/splits/train_patients.txt` and `data/splits/val_patients.txt`).

In [ ]:
processed_dir = os.path.join(PROJECT_ROOT, config['data']['processed_dir'])
train_split = os.path.join(PROJECT_ROOT, config['data']['train_split'])
val_split = os.path.join(PROJECT_ROOT, config['data']['val_split'])

train_dataset = ACDCSegDataset(
    processed_dir=processed_dir,
    split_file=train_split,
    transform=get_train_transforms(),
)
val_dataset = ACDCSegDataset(
    processed_dir=processed_dir,
    split_file=val_split,
    transform=get_val_transforms(),
)

print(f"Training set (labeled slices):   {len(train_dataset)}")
print(f"Validation set (labeled slices): {len(val_dataset)}")

# Load one sample batch
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=False)
batch = next(iter(train_loader))
images = batch['image'].to(DEVICE)
masks = batch['mask'].to(DEVICE)

print(f"\nSample batch image tensor: {images.shape} ({images.dtype})")
print(f"Sample batch mask tensor:  {masks.shape} ({masks.dtype})")
print(f"Mask unique labels:        {torch.unique(masks).tolist()}")

## 2. Model Architecture & Instantiation

Instantiates the 2D U-Net with a 4-level shared encoder (`[32, 64, 128, 256]` channels) and symmetric decoder.

In [ ]:
model = build_segmentation_model(config).to(DEVICE)
total_params = count_parameters(model)

print(f"Model Architecture: {model.__class__.__name__}")
print(f"Input channels:     {config['model']['in_channels']}")
print(f"Output classes:     {config['model']['num_classes']}")
print(f"Encoder channels:   {config['model']['encoder_channels']}")
print(f"Total Parameters:   {total_params:,} ({total_params / 1e6:.2f} M)")

## 3. Forward Pass & Output Logits

Verifies that a 2D MRI slice of dimensions `(B, 1, 256, 256)` produces logits of shape `(B, 4, 256, 256)`.

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(images)
    probs = F.softmax(logits, dim=1)

print(f"Logits tensor shape:      {logits.shape} (B, num_classes, H, W)")
print(f"Probabilities shape:      {probs.shape}")
print(f"Probabilities sum to 1.0: {torch.allclose(probs.sum(dim=1), torch.ones_like(probs.sum(dim=1)))}")

## 4. Loss Function Computation

Computes the combined **Dice + Cross-Entropy Loss** (${\mathcal{L}}_{\text{DiceCE}} = \lambda_{\text{dice}} {\mathcal{L}}_{\text{Dice}} + \lambda_{\text{ce}} {\mathcal{L}}_{\text{CE}}$).

In [ ]:
criterion = DiceCELoss(
    num_classes=config['model']['num_classes'],
    dice_weight=config['loss']['dice_weight'],
    ce_weight=config['loss']['ce_weight'],
    include_background=config['loss']['include_background'],
)

loss = criterion(logits, masks)
print(f"DiceCELoss value: {loss.item():.4f}")
assert torch.isfinite(loss), "Loss must be a finite scalar!"

## 5. Metric Calculation

Calculates per-class Dice scores (LV, Myocardium, RV), mean Dice, and 95th-percentile Hausdorff Distance (HD95).

In [ ]:
preds = torch.argmax(logits, dim=1).cpu().numpy()
targets = masks.cpu().numpy()

sample_metrics = compute_metrics_single(preds[0], targets[0], compute_hd=True)
print("Evaluation Metrics (Untrained Baseline Sample):")
print(f"  - LV Dice:         {sample_metrics['LV_Dice']:.4f}")
print(f"  - Myocardium Dice: {sample_metrics['Myocardium_Dice']:.4f}")
print(f"  - RV Dice:         {sample_metrics['RV_Dice']:.4f}")
print(f"  - Mean Dice:       {sample_metrics['Mean_Dice']:.4f}")
print(f"  - Mean HD95:       {sample_metrics['Mean_HD95']:.2f} mm")

## 6. Qualitative Visualization

Visualizes the input slice, ground-truth segmentation, and initial model prediction.

In [ ]:
# Custom colormap for 4 classes
# 0: BG (Black), 1: LV (Red), 2: Myo (Green), 3: RV (Blue)
cmap_classes = ListedColormap(['black', '#e41a1c', '#4daf4a', '#377eb8'])

img_np = images[0, 0].cpu().numpy()
gt_np = targets[0]
pred_np = preds[0]
pid = batch['patient_id'][0]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle(f"Baseline Cardiac Segmentation Verification — Patient {pid}", fontsize=14, fontweight='bold')

axes[0].imshow(img_np, cmap='gray')
axes[0].set_title("Normalized Cine MRI (256x256)")
axes[0].axis('off')

axes[1].imshow(gt_np, cmap=cmap_classes, vmin=0, vmax=3)
axes[1].set_title("Ground Truth Mask\n(1=LV, 2=Myo, 3=RV)")
axes[1].axis('off')

axes[2].imshow(pred_np, cmap=cmap_classes, vmin=0, vmax=3)
axes[2].set_title("Model Raw Prediction\n(Untrained Weights)")
axes[2].axis('off')

axes[3].imshow(img_np, cmap='gray')
axes[3].imshow(np.ma.masked_where(gt_np == 0, gt_np), cmap=cmap_classes, alpha=0.5, vmin=0, vmax=3)
axes[3].set_title("GT Mask Overlay on MRI")
axes[3].axis('off')

plt.tight_layout()
figures_dir = os.path.join(PROJECT_ROOT, 'results', 'figures')
os.makedirs(figures_dir, exist_ok=True)
fig_path = os.path.join(figures_dir, 'baseline_prediction_example.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f"Saved verification figure to: {fig_path}")
plt.show()

## 7. Separate GPU Training System Instructions

> **Note**: This notebook establishes that all modules, loss functions, metrics, and dataset loaders function correctly. When transitioning to the training GPU server, execute the full training script as documented below:

In [ ]:
train_cmd = "python src/train.py --config configs/baseline_config.yaml --device cuda"
print("=" * 70)
print("COMMAND TO EXECUTE ON THE GPU TRAINING MACHINE:")
print("=" * 70)
print(f"\n  {train_cmd}\n")
print("Expected output checkpoints: checkpoints/baseline_unet_best.pth")
print("=" * 70)